# Demo: a post's topic and sentiment from its comments

Classifies every comment under a social media post with a model of this project, and reads the post's
**topic** and **sentiment** from all its comments together (the mean of the comments' probabilities).
Comments can be pasted (Facebook, X, TikTok, forums…) or read from a **YouTube** link.

**Kaggle notebook settings**
- **Accelerator:** GPU **T4**.
- **Internet: On** (the code is cloned from GitHub; the encoder and YouTube need the internet).

**How to run:** edit section 1 if needed, then *Run All* in the editor. The first time there is no model, so
section 4 trains the two demo models (about 1 GPU-hour) and keeps their weights in `/kaggle/working/demo_checkpoints`.
To skip training next time, *Save Version* once, then in a new session use **Add Input → Notebook Output**
on that version: section 4 finds the weights in `/kaggle/input` by itself.

The models learned from Vietnamese student forum posts (NEU-ESC) or course feedback (UIT-VSFC). Comments in other
languages are left out; comments on other subjects are outside what they learned, so treat the answers as a rough guide.

## 1. Settings
The only cell to edit.

In [ ]:
GIT_URL = 'https://github.com/Zaidve/DPL_fall36_project.git'   # None = use an attached Kaggle Dataset instead
GIT_BRANCH = 'main'
CHECKPOINTS = None        # folder with run folders (config.yaml + best.pt), e.g. '/kaggle/input/<output>/models'; None = search
TRAIN_IF_MISSING = True   # no model found: train the demo models (configs/experiment/demo_model.yaml, about 1 GPU-hour)
TRAIN_ONLY = None         # e.g. 'neu-esc__' to train only the NEU-ESC model (about 45 min)
MODEL = None              # run id to use, e.g. 'uit-vsfc__phobert__mtl__seed42__sum'; None = the NEU-ESC model
YOUTUBE_SECRET = 'YOUTUBE_API_KEY'   # name of the Kaggle secret holding a YouTube Data API key ('' = no YouTube)
START_WEB_PAGE = False    # serve the demo page through a public tunnel (read section 8 first)
ZIP_CHECKPOINTS = False   # write /kaggle/working/demo_checkpoints.zip, to run the demo on your own computer

## 2. Find the code
Clones `GIT_URL` into `/kaggle/working` (or finds the project in the attached Datasets). Locally: the project
this notebook is in.

In [ ]:
import os, sys, shutil, subprocess, zipfile
from pathlib import Path

ON_KAGGLE = os.path.isdir('/kaggle/input')
WORK = Path('/kaggle/working')


def is_project(p):
    return (Path(p) / 'trainer' / 'train.py').is_file() and (Path(p) / 'demo' / 'server.py').is_file()


if ON_KAGGLE and GIT_URL:
    CODE = WORK / 'DPL_project'
    if CODE.exists():
        subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '-b', GIT_BRANCH, GIT_URL, str(CODE)], check=True)
elif ON_KAGGLE:
    CODE = next((p.parent.parent for p in Path('/kaggle/input').glob('**/demo/server.py') if is_project(p.parent.parent)), None)
    if CODE is None:
        raise FileNotFoundError('Project not found in /kaggle/input: attach it as a Dataset or set GIT_URL.')
else:
    CODE = next((p for p in (Path.cwd(), *Path.cwd().parents[:2]) if is_project(p)), None)
    if CODE is None:
        raise FileNotFoundError('Run this notebook from inside the project.')

os.chdir(CODE)
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
sys.dont_write_bytecode = True
os.environ.setdefault('DPL_PROCESSED_DIR', str(CODE / 'data' / 'processed'))
DEMO_MODELS = WORK / 'demo_checkpoints' if ON_KAGGLE else CODE / 'demo' / 'checkpoints'
print('code  :', CODE)
print('models:', DEMO_MODELS)

## 3. Packages and GPU
Kaggle already has PyTorch and transformers. The demo also needs `sentencepiece` and `underthesea`
(PhoBERT's word segmentation).

In [ ]:
if ON_KAGGLE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece', 'underthesea'], check=True)

import torch, transformers
print(f'torch {torch.__version__} | transformers {transformers.__version__}')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU: predicting works on the CPU (slower); training does not. Set Accelerator = GPU T4 to train.')

## 4. Find the models, or train them
A model is a run folder with `config.yaml` and `best.pt`. This cell looks in `CHECKPOINTS`, in
`/kaggle/working/demo_checkpoints`, `/kaggle/working/models`, and in every `models` folder of the attached inputs.
If there is none and `TRAIN_IF_MISSING` is on, it trains the demo models (seed 42, weights kept):

| Run folder | Model |
|---|---|
| `neu-esc__phobert__mtl__seed42__smartref` | PhoBERT, linear heads, reference SMART: 4 sentiments, 10 topics of student forum posts |
| `uit-vsfc__phobert__mtl__seed42__sum` | PhoBERT, linear heads, equal sum: 3 sentiments, 4 topics of course feedback |

In [ ]:
from demo.predictor import find_run_dirs


def search_models():
    if CHECKPOINTS:
        return find_run_dirs([CHECKPOINTS])
    roots = [DEMO_MODELS, WORK / 'models']
    if ON_KAGGLE:
        for pattern in ('*', '*/models', '*/*/models', '*/*/*/models', '*/*/*/*/models'):
            roots += sorted(Path('/kaggle/input').glob(pattern))
    found, seen = [], set()
    for d in find_run_dirs([r for r in roots if r.is_dir()]):
        if d.name not in seen:
            seen.add(d.name)
            found.append(d)
    return found


def run(cmd, env=None):
    proc = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            encoding='utf-8', errors='replace', env={**os.environ, 'PYTHONIOENCODING': 'utf-8', **(env or {})})
    for line in proc.stdout:
        if 'Loading weights' not in line:
            print(line, end='')
    if proc.wait() != 0:
        raise RuntimeError(f'command failed ({proc.returncode}): {" ".join(cmd)}')


run_dirs = search_models()
if not run_dirs and TRAIN_IF_MISSING:
    if not torch.cuda.is_available():
        raise RuntimeError('No model found and no GPU to train one: set Accelerator = GPU T4, or attach an output with models.')
    print(f'No model found: training the demo models into {DEMO_MODELS} (about 1 GPU-hour) ...')
    cmd = [sys.executable, '-m', 'trainer.train', '--experiment', 'configs/experiment/demo_model.yaml']
    cmd += (['--only', TRAIN_ONLY] if TRAIN_ONLY else []) + (['--time-budget-hours', '9.5'] if ON_KAGGLE else [])
    run(cmd, env={'DPL_MODELS_DIR': str(DEMO_MODELS)})
    run_dirs = search_models()
if not run_dirs:
    raise FileNotFoundError('No model (config.yaml + best.pt) found. Set TRAIN_IF_MISSING = True or CHECKPOINTS.')
run_dirs.sort(key=lambda d: (not d.name.startswith('neu-esc'), d.name))
for d in run_dirs:
    print('found', d)

## 5. Load a model
`MODEL` picks a run id; by default the NEU-ESC model, whose topics come from student forum posts.

In [ ]:
import html
import pandas as pd
from IPython.display import HTML, display
from demo.core import split_comments
from demo.predictor import DISPLAY, Predictor
from demo.server import analyze

chosen = next((d for d in run_dirs if d.name == MODEL), None) if MODEL else run_dirs[0]
if chosen is None:
    raise KeyError(f'MODEL {MODEL!r} not found; available: {[d.name for d in run_dirs]}')
predictor = Predictor.from_run_dir(chosen)
info = predictor.describe()
print(f"model: {info['id']}  ({info['dataset_name']}, {info['backbone']}, {info['mode']}, {info['tag']})")


def _bars(task, agg):
    rows = sorted(agg['distribution'].items(), key=lambda kv: -kv[1])
    bars = ''.join(
        f'<div style="display:flex;align-items:center;gap:8px;font-size:13px;margin:2px 0">'
        f'<span style="width:130px">{html.escape(DISPLAY.get(n, n))}</span>'
        f'<span style="flex:1;background:#e5e9f0;height:9px;border-radius:5px;overflow:hidden">'
        f'<span style="display:block;height:100%;width:{100 * p:.1f}%;background:#2a64d6"></span></span>'
        f'<span style="width:42px;text-align:right">{p:.0%}</span></div>' for n, p in rows)
    return (f'<div style="flex:1;min-width:260px"><div style="font-size:13px;color:#666">{task.title()}</div>'
            f'<div style="font-size:22px;font-weight:700">{html.escape(DISPLAY.get(agg["label"], agg["label"]))}</div>'
            f'<div style="font-size:12px;color:#666;margin-bottom:6px">{agg["probability"]:.0%} average probability · '
            f'{agg["agreement"]:.0%} of comments agree ({agg["counts"][agg["label"]]} of {agg["n"]})</div>{bars}</div>')


def show(result):
    """ The post-level answer as bars, then one row per comment. """
    boxes = ''.join(_bars(t, result['post'][t]) if result['post'][t] else
                    f'<div style="flex:1">{t}: no Vietnamese comment to use</div>' for t in result['tasks'])
    notes = []
    if result['n_other_language']:
        notes.append(f"{result['n_other_language']} comment(s) do not look Vietnamese and were left out.")
    if result.get('post_text'):
        pt = result['post_text']
        notes.append('The post text alone gives ' + ', '.join(
            f"{t}: {DISPLAY.get(pt[t]['label'], pt[t]['label'])} ({pt[t]['probability']:.0%})" for t in result['tasks'] if pt[t]) + '.')
    display(HTML(f'<div style="font-weight:600;margin:6px 0">The post, read from {result["n_used"]} of '
                 f'{result["n_comments"]} comments</div><div style="display:flex;gap:24px;flex-wrap:wrap">{boxes}</div>'
                 + ''.join(f'<p style="font-size:13px;color:#8a5a00">{html.escape(n)}</p>' for n in notes)))
    table = pd.DataFrame([{'comment': c['text'], 'used': c['used'],
                           **{t: f"{DISPLAY.get(c[t]['label'], c[t]['label'])} ({c[t]['probability']:.0%})" if c[t] else ''
                              for t in result['tasks']}} for c in result['comments']])
    with pd.option_context('display.max_colwidth', 120, 'display.max_rows', 200):
        display(table)


def analyze_comments(comments, post_text=None, include_other_languages=False):
    result = analyze(predictor, comments, post_text, include_other_languages)
    show(result)
    return result

## 6. Try it

In [ ]:
example = [
    'Thầy dạy dễ hiểu lắm, cảm ơn thầy!',
    'Môn này khó quá, ai có tài liệu cho mình xin với',
    'ko biết thi cuối kì có khó ko mn ơi',
    'Đề thi năm nay khó thật sự, học mãi không vào',
    'This course is great, highly recommend',
]
_ = analyze_comments(example, post_text='Mọi người thấy môn Kinh tế vi mô kỳ này thế nào?')

## 7. Your own comments
Paste the comments between the triple quotes: one comment per line (`SPLIT = 'line'`), or separated by empty lines
for comments that span several lines (`SPLIT = 'blank'`). `POST_TEXT` is optional.

In [ ]:
COMMENTS_TEXT = """
Paste the comments here, one per line
"""
SPLIT = 'line'
POST_TEXT = ''
INCLUDE_OTHER_LANGUAGES = False

comments = [c for c in split_comments(COMMENTS_TEXT, SPLIT) if c != 'Paste the comments here, one per line']
if not comments:
    print('Paste comments into COMMENTS_TEXT, then run this cell again.')
else:
    print(len(comments), 'comments')
    _ = analyze_comments(comments, POST_TEXT, INCLUDE_OTHER_LANGUAGES)

## 8. A YouTube video
Needs a free YouTube Data API key (Google Cloud console → enable **YouTube Data API v3** → **Credentials → API key**).
Store it as a Kaggle secret: **Add-ons → Secrets**, name `YOUTUBE_API_KEY` (the name in `YOUTUBE_SECRET`), and tick it
for this notebook. The key never appears in the notebook. Each 100 comments cost 1 unit of the free 10,000 a day.

In [ ]:
from demo.server import YouTubeAPI, fetch_comments

youtube = None
if YOUTUBE_SECRET:
    try:
        from kaggle_secrets import UserSecretsClient
        youtube = YouTubeAPI(UserSecretsClient().get_secret(YOUTUBE_SECRET))
    except Exception as e:  # noqa: BLE001
        print(f'No YouTube key ({type(e).__name__}): add the secret {YOUTUBE_SECRET!r} under Add-ons -> Secrets.')
else:
    youtube = YouTubeAPI(os.environ.get('YOUTUBE_API_KEY', ''))

VIDEO_URL = 'https://www.youtube.com/watch?v=VIDEO_ID'

if 'VIDEO_ID' in VIDEO_URL:
    print('Set VIDEO_URL to a YouTube link, then run this cell again.')
else:
    thread = fetch_comments(VIDEO_URL, youtube=youtube)
    if 'error' in thread:
        print(thread['error'])
    else:
        print(f"{thread['post']['title']} ({thread['post']['source']}): {len(thread['comments'])} comments")
        _ = analyze_comments(thread['comments'], post_text=thread['post']['title'])

## 9. The web page (optional)
With `START_WEB_PAGE = True` this serves the demo page and opens a **public** `https://….trycloudflare.com` address
with cloudflared, so you can use the page in your browser while the model runs on Kaggle. **Anyone with the address
can use it and there is no password:** use it only for a short live demo, then run the next cell to stop it, and
check Kaggle's rules first. The address works only while this session runs (interactive editor, not *Save Version*).

In [ ]:
import re, threading, time
from demo.server import Models, ThreadingHTTPServer, make_handler

web, tunnel = None, None
if START_WEB_PAGE:
    models = Models([chosen], loader=lambda d: predictor)
    web = ThreadingHTTPServer(('127.0.0.1', 8765), make_handler(models, youtube=youtube))
    threading.Thread(target=web.serve_forever, daemon=True).start()
    binary = WORK / 'cloudflared'
    if not binary.exists():
        subprocess.run(['wget', '-q', '-O', str(binary),
                        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
        binary.chmod(0o755)
    tunnel = subprocess.Popen([str(binary), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8765'],
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    deadline, url = time.time() + 60, None
    while time.time() < deadline and url is None:
        line = tunnel.stdout.readline()
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line or '')
        url = m.group(0) if m else None
    print('demo page:', url or 'no address yet; check that Internet is On')
else:
    print('START_WEB_PAGE is False: the web page is not started.')

In [ ]:
# stop the web page and the tunnel
if tunnel is not None:
    tunnel.terminate()
if web is not None:
    web.shutdown()
    web.server_close()
web, tunnel = None, None
print('stopped')

## 10. Take the models home (optional)
With `ZIP_CHECKPOINTS = True` this writes `/kaggle/working/demo_checkpoints.zip` (`config.yaml` + `best.pt` of each
model, about 270 MB each). Download it from the Output panel and unzip it into `demo/checkpoints/` of the project on your
computer; then `python -m demo.server` shows the same page there, with no public address.

In [ ]:
if ZIP_CHECKPOINTS:
    out = (WORK if ON_KAGGLE else CODE) / 'demo_checkpoints.zip'
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_STORED) as z:     # fp16 weights do not compress
        for d in run_dirs:
            for name in ('config.yaml', 'best.pt', 'metrics.json'):
                if (d / name).is_file():
                    z.write(d / name, f'{d.name}/{name}')
    print(f'{out} ({out.stat().st_size / 2**20:.0f} MB): unzip into demo/checkpoints/')
else:
    print('ZIP_CHECKPOINTS is False.')